# Global Solution — CitizenTech
## Mobilidade Urbana Inteligente: Modelagem Matemática para a Gestão do Transporte Público

**Disciplina:** Modelagem Matemática e Computacional — FIAP

**Integrantes:**

| Nome | RM |
|---|---|
| Ian Rodrigues Martins | 570540 |
| Patrick Fernandes Martins | RM_PATRICK |
| Gabriel Del Pizzo Pintor | RM_GABRIEL |

**Ferramentas:** Python + NumPy (sistemas lineares, matrizes e transformações lineares)

# ETAPA 1 — Diagnóstico e modelagem da situação atual

### a) Variáveis
Cada variável representa a quantidade de ônibus alocada hoje em um centro operacional:

| Variável | Significado |
|---|---|
| $N$ | ônibus da região Norte |
| $S$ | ônibus da região Sul |
| $L$ | ônibus da região Leste |
| $O$ | ônibus da região Oeste |

Restrições: $N, S, L, O \in \mathbb{Z}$ e $N, S, L, O \ge 0$.

### b) Modelo matemático (Relatório I)
1. Frota total: $N + S + L + O = 72$
2. Norte + Oeste + 2·Sul: $N + 2S + O = 76$
3. Sul + 2·Leste: $S + 2L = 52$
4. Norte + Leste + 2·Oeste: $N + L + 2O = 76$

### c) Forma matricial $A\mathbf{x} = \mathbf{b}$

$$
\underbrace{\begin{bmatrix}
1 & 1 & 1 & 1\\
1 & 2 & 0 & 1\\
0 & 1 & 2 & 0\\
1 & 0 & 1 & 2
\end{bmatrix}}_{A}
\underbrace{\begin{bmatrix} N\\ S\\ L\\ O \end{bmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{bmatrix} 72\\ 76\\ 52\\ 76 \end{bmatrix}}_{\mathbf{b}}
$$

In [1]:
import numpy as np
from fractions import Fraction

regioes = ["Norte", "Sul", "Leste", "Oeste"]

A = np.array([
    [1, 1, 1, 1],   # N + S + L + O      = 72
    [1, 2, 0, 1],   # N + 2S + O         = 76
    [0, 1, 2, 0],   # S + 2L             = 52
    [1, 0, 1, 2],   # N + L + 2O         = 76
], dtype=float)

b = np.array([72, 76, 52, 76], dtype=float)

print("Matriz A:\n", A.astype(int))
print("Vetor b:", b.astype(int))

Matriz A:
 [[1 1 1 1]
 [1 2 0 1]
 [0 1 2 0]
 [1 0 1 2]]
Vetor b: [72 76 52 76]


### d) O sistema tem solução única?

Pelo **Teorema de Rouché–Capelli**, um sistema $A\mathbf{x}=\mathbf{b}$ com $n$ incógnitas:
- é **possível e determinado** (solução única) se $\text{posto}(A) = \text{posto}([A|\mathbf{b}]) = n$;
- é **possível e indeterminado** se os postos são iguais mas menores que $n$;
- é **impossível** se os postos são diferentes.

Para uma matriz quadrada, isso equivale a $\det(A) \neq 0$ (matriz **invertível**, colunas linearmente independentes).

**Cálculo do determinante** (expansão de Laplace / escalonamento):

Escalonando $A$:
- $L_2 \leftarrow L_2 - L_1$: $[0, 1, -1, 0]$
- $L_4 \leftarrow L_4 - L_1$: $[0, -1, 0, 1]$
- $L_3 \leftarrow L_3 - L_2$: $[0, 0, 3, 0]$
- $L_4 \leftarrow L_4 + L_2$: $[0, 0, -1, 1]$
- $L_4 \leftarrow L_4 + \tfrac{1}{3}L_3$: $[0, 0, 0, 1]$

Matriz triangular com diagonal $(1, 1, 3, 1)$ ⇒ $\det(A) = 1\cdot1\cdot3\cdot1 = 3 \neq 0$.

Logo $\text{posto}(A) = 4$, $A$ é invertível e o sistema tem **uma única solução**: $\mathbf{x} = A^{-1}\mathbf{b}$. As informações do Relatório I são suficientes para reconstruir a frota.

In [2]:
det_A = np.linalg.det(A)
posto_A = np.linalg.matrix_rank(A)
posto_Ab = np.linalg.matrix_rank(np.column_stack([A, b]))

print(f"det(A)          = {det_A:.4f}")
print(f"posto(A)        = {posto_A}")
print(f"posto([A | b])  = {posto_Ab}")
print(f"nº de incógnitas = {A.shape[1]}")

if posto_A == posto_Ab == A.shape[1]:
    print("=> Sistema possível e DETERMINADO: existe uma única distribuição de ônibus.")

det(A)          = 3.0000
posto(A)        = 4
posto([A | b])  = 4
nº de incógnitas = 4
=> Sistema possível e DETERMINADO: existe uma única distribuição de ônibus.


### e) Resolução computacional

Resolvendo à mão por substituição:
- (2) − (1): $S - L = 4 \Rightarrow S = L + 4$
- Em (3): $(L+4) + 2L = 52 \Rightarrow L = 16$, logo $S = 20$
- Em (1): $N + O = 36$; em (4): $N + 2O = 60 \Rightarrow O = 24$, $N = 12$

Conferindo com Python:

In [3]:
x_atual = np.linalg.solve(A, b)
x_atual = np.rint(x_atual).astype(int)   # arredonda erros de ponto flutuante

print("Distribuição atual da frota:")
for r, q in zip(regioes, x_atual):
    print(f"  {r:<6}: {q:>3} ônibus")
print(f"  Total : {x_atual.sum():>3} ônibus")

# Solução também via matriz inversa (mostra que A é invertível)
A_inv = np.linalg.inv(A)
print("\nA^-1 (x3, para visualizar em inteiros):\n", np.rint(A_inv * 3).astype(int))
print("x = A^-1 · b =", np.rint(A_inv @ b).astype(int))

Distribuição atual da frota:
  Norte :  12 ônibus
  Sul   :  20 ônibus
  Leste :  16 ônibus
  Oeste :  24 ônibus
  Total :  72 ônibus

A^-1 (x3, para visualizar em inteiros):
 [[ 9 -3 -3 -3]
 [-2  2  1  0]
 [ 1 -1  1  0]
 [-5  2  1  3]]
x = A^-1 · b = [12 20 16 24]


### f) Verificação dos registros operacionais

In [4]:
N, S, L, O = x_atual
verificacoes = [
    ("Frota total = 72",          N + S + L + O, 72),
    ("N + O + 2S = 76",           N + O + 2*S,   76),
    ("S + 2L = 52",               S + 2*L,       52),
    ("N + L + 2O = 76",           N + L + 2*O,   76),
    ("Valores inteiros e >= 0",   int(all(v >= 0 for v in x_atual)), 1),
]
for desc, obtido, esperado in verificacoes:
    print(f"{'OK ' if obtido == esperado else 'ERRO'} | {desc:<26} -> obtido {obtido}, esperado {esperado}")

print("\nA·x == b ?", np.allclose(A @ x_atual, b))

OK  | Frota total = 72           -> obtido 72, esperado 72
OK  | N + O + 2S = 76            -> obtido 76, esperado 76
OK  | S + 2L = 52                -> obtido 52, esperado 52
OK  | N + L + 2O = 76            -> obtido 76, esperado 76
OK  | Valores inteiros e >= 0    -> obtido 1, esperado 1

A·x == b ? True


**Conclusão da Etapa 1:** a frota atual é **Norte = 12, Sul = 20, Leste = 16, Oeste = 24** (total 72), única solução que satisfaz todos os registros.

---
# ETAPA 2 — Política de redistribuição

### a) Ônibus necessários por região
Necessidade $= \left\lceil \dfrac{\text{passageiros}}{\text{capacidade}} \right\rceil$ (arredonda para cima: nenhum passageiro pode ficar sem vaga).

In [5]:
passageiros = np.array([560, 760, 720, 840])
capacidade  = np.array([40, 40, 40, 40])

necessidade = np.ceil(passageiros / capacidade).astype(int)

print(f"{'Região':<7}{'Passag.':>9}{'Cap.':>6}{'Necessários':>13}")
for r, p, c, n in zip(regioes, passageiros, capacidade, necessidade):
    print(f"{r:<7}{p:>9}{c:>6}{n:>13}")
print(f"{'Total':<7}{passageiros.sum():>9}{'':>6}{necessidade.sum():>13}")
print(f"\nFrota disponível: {x_atual.sum()}  |  Necessidade total: {necessidade.sum()}")

Região   Passag.  Cap.  Necessários
Norte        560    40           14
Sul          760    40           19
Leste        720    40           18
Oeste        840    40           21
Total       2880                 72

Frota disponível: 72  |  Necessidade total: 72


A necessidade total (72) é **exatamente igual** à frota disponível. Portanto é possível atender 100% da demanda **sem ampliar a frota**, apenas redistribuindo.

### b) Desequilíbrios

In [6]:
saldo = x_atual - necessidade   # >0 sobra, <0 falta

print(f"{'Região':<7}{'Atual':>7}{'Necess.':>9}{'Saldo':>7}  Situação")
for r, a, n, s in zip(regioes, x_atual, necessidade, saldo):
    sit = "excedente" if s > 0 else ("déficit" if s < 0 else "equilibrada")
    print(f"{r:<7}{a:>7}{n:>9}{s:>+7}  {sit}")

# Situação atual de atendimento
atendidos = np.minimum(x_atual * capacidade, passageiros)
print(f"\nPassageiros sem vaga hoje: {(passageiros - atendidos).sum()} "
      f"(Norte: {passageiros[0]-atendidos[0]}, Leste: {passageiros[2]-atendidos[2]})")
print(f"Vagas ociosas hoje: {(x_atual*capacidade - atendidos).sum()} (Sul + Oeste)")

Região   Atual  Necess.  Saldo  Situação
Norte       12       14     -2  déficit
Sul         20       19     +1  excedente
Leste       16       18     -2  déficit
Oeste       24       21     +3  excedente

Passageiros sem vaga hoje: 160 (Norte: 80, Leste: 80)
Vagas ociosas hoje: 160 (Sul + Oeste)


**Diagnóstico:**
- **Norte**: faltam 2 ônibus (80 passageiros sem vaga)
- **Leste**: faltam 2 ônibus (80 passageiros sem vaga)
- **Sul**: sobra 1 ônibus
- **Oeste**: sobram 3 ônibus

### c) Política proposta

**Critérios adotados:**
1. **Atendimento total:** cada região passa a ter exatamente o número de ônibus necessário.
2. **Mínimo de movimentações:** só se move a quantidade de ônibus em excesso (4 no total). Regiões com déficit não cedem veículos.
3. **Proximidade geográfica:** os ônibus vão de centros vizinhos (Oeste → Norte, Oeste → Leste, Sul → Leste), reduzindo deslocamento vazio.
4. **Frota fixa:** nenhum ônibus é criado ou retirado.

| Origem → Destino | Ônibus |
|---|---|
| Oeste → Norte | 2 |
| Oeste → Leste | 1 |
| Sul → Leste | 1 |

### d) e e) Transformação linear e sua matriz

Definimos $T:\mathbb{R}^4 \to \mathbb{R}^4$, $T(\mathbf{x}) = M\mathbf{x}$, em que $\mathbf{x} = (N, S, L, O)^T$.

O coeficiente $m_{ij}$ é **a fração dos ônibus da região $j$ (coluna, origem) que passa a operar na região $i$ (linha, destino)**. A diagonal $m_{jj}$ é a fração que permanece na própria região.

Frações da política:
- Norte e Leste não cedem: mantêm 100%.
- Sul (20 ônibus) cede 1 ao Leste: fica com $19/20$, envia $1/20$.
- Oeste (24 ônibus) cede 2 ao Norte e 1 ao Leste: fica com $21/24 = 7/8$, envia $2/24 = 1/12$ ao Norte e $1/24$ ao Leste.

$$
M = \begin{bmatrix}
1 & 0 & 0 & \tfrac{1}{12}\\[2pt]
0 & \tfrac{19}{20} & 0 & 0\\[2pt]
0 & \tfrac{1}{20} & 1 & \tfrac{1}{24}\\[2pt]
0 & 0 & 0 & \tfrac{7}{8}
\end{bmatrix}
\qquad
\begin{array}{l}
\text{linhas = destino}\\
\text{colunas = origem}\\
\text{ordem: } N, S, L, O
\end{array}
$$

Exemplo de leitura: $m_{14} = 1/12$ significa que 1/12 dos ônibus do Oeste vão para o Norte ($24 \cdot 1/12 = 2$).

In [7]:
F = Fraction
M_frac = [
    [F(1), F(0),     F(0), F(1, 12)],
    [F(0), F(19, 20), F(0), F(0)    ],
    [F(0), F(1, 20), F(1), F(1, 24)],
    [F(0), F(0),     F(0), F(7, 8)  ],
]
M = np.array([[float(v) for v in linha] for linha in M_frac])

print("Matriz M (frações exatas):")
for r, linha in zip(regioes, M_frac):
    print(f"  {r:<6}", "  ".join(f"{str(v):>6}" for v in linha))

# Aplicação da transformação com aritmética exata
x_novo = [sum(M_frac[i][j] * int(x_atual[j]) for j in range(4)) for i in range(4)]
x_novo = np.array([int(v) for v in x_novo])

print("\nT(x_atual) = M · x_atual")
print(f"{'Região':<7}{'Antes':>7}{'Depois':>8}{'Necess.':>9}")
for r, a, d, n in zip(regioes, x_atual, x_novo, necessidade):
    print(f"{r:<7}{a:>7}{d:>8}{n:>9}")
print(f"{'Total':<7}{x_atual.sum():>7}{x_novo.sum():>8}{necessidade.sum():>9}")
print("\nNova distribuição atende toda a demanda?", np.array_equal(x_novo, necessidade))

Matriz M (frações exatas):
  Norte       1       0       0    1/12
  Sul         0   19/20       0       0
  Leste       0    1/20       1    1/24
  Oeste       0       0       0     7/8

T(x_atual) = M · x_atual
Região   Antes  Depois  Necess.
Norte       12      14       14
Sul         20      19       19
Leste       16      18       18
Oeste       24      21       21
Total       72      72       72

Nova distribuição atende toda a demanda? True


### f) Demonstração: conservação do total e não negatividade

**1) Linearidade.** $T(\mathbf{x}) = M\mathbf{x}$ é produto matriz–vetor, logo $T(\alpha\mathbf{x} + \beta\mathbf{y}) = \alpha T(\mathbf{x}) + \beta T(\mathbf{y})$.

**2) Conservação do total.** Seja $\mathbf{1} = (1,1,1,1)^T$. O total de ônibus é $\mathbf{1}^T\mathbf{x}$.
Cada coluna de $M$ soma 1:
- Col. N: $1 + 0 + 0 + 0 = 1$
- Col. S: $0 + \tfrac{19}{20} + \tfrac{1}{20} + 0 = 1$
- Col. L: $0 + 0 + 1 + 0 = 1$
- Col. O: $\tfrac{1}{12} + 0 + \tfrac{1}{24} + \tfrac{7}{8} = \tfrac{2 + 1 + 21}{24} = 1$

Isso significa $\mathbf{1}^T M = \mathbf{1}^T$. Então, para qualquer $\mathbf{x}$:
$$
\mathbf{1}^T T(\mathbf{x}) = \mathbf{1}^T M \mathbf{x} = \mathbf{1}^T \mathbf{x}
$$
ou seja, **o total de ônibus antes e depois é o mesmo** (72 = 72). Intuitivamente: cada ônibus de uma região é 100% distribuído entre as regiões, sem sobra nem criação.

**3) Não negatividade.** Todos os $m_{ij} \ge 0$. Se $\mathbf{x} \ge 0$, cada componente
$$
T(\mathbf{x})_i = \sum_j m_{ij}\, x_j
$$
é soma de produtos de números não negativos, logo $T(\mathbf{x})_i \ge 0$. **Nenhuma alocação negativa é produzida.**

(Uma matriz com entradas $\ge 0$ e colunas somando 1 é chamada **matriz estocástica por colunas** — exatamente o tipo de matriz que modela redistribuição sem perda.)

**4) Integralidade.** Para a frota atual, $M\mathbf{x} = (14, 19, 18, 21)$, todos inteiros, como exigido.

**Por que não usar simplesmente $T(\mathbf{x}) = \mathbf{x} + \Delta$, com $\Delta = (2, -1, 2, -3)$?** Porque isso é uma **translação**, não uma transformação linear: $T(\mathbf{0}) = \Delta \ne \mathbf{0}$, e uma transformação linear sempre leva o vetor nulo nele mesmo. Além disso, aplicada a uma frota pequena, ela geraria valores negativos. A matriz $M$ evita os dois problemas, porque redistribui **frações** da frota de cada região.

**Limitação:** os coeficientes de $M$ foram calibrados para a frota atual $(12, 20, 16, 24)$. Para ela o resultado é inteiro. Aplicada a outra distribuição, $M\mathbf{x}$ continua conservando o total e sem negativos, mas pode gerar valores fracionários, que precisariam ser arredondados mantendo a soma.

*Observação:* $\det(M) = \tfrac{19}{20}\cdot\tfrac{7}{8} \ne 0$, então $M$ é invertível; a política pode ser desfeita aplicando $M^{-1}$, voltando à distribuição original.

In [8]:
uns = np.ones(4)

print("Soma de cada coluna de M:", [str(sum(col)) for col in zip(*M_frac)])
print("1^T · M == 1^T ?", np.allclose(uns @ M, uns))
print("Todas as entradas de M >= 0 ?", bool((M >= 0).all()))
print(f"Total antes: {x_atual.sum()}  |  Total depois: {x_novo.sum()}")
print("Nova alocação toda >= 0 e inteira ?", bool((x_novo >= 0).all()))

# Teste extra: conservação vale para QUALQUER distribuição não negativa
rng = np.random.default_rng(42)
for _ in range(3):
    x_teste = rng.integers(0, 50, size=4)
    y = M @ x_teste
    print(f"  x={x_teste}, soma={x_teste.sum()} -> soma de Mx = {y.sum():.4f}, min(Mx) = {y.min():.3f}")

print(f"\ndet(M) = {np.linalg.det(M):.4f}  (invertível)")
print("M^-1 · x_novo =", np.rint(np.linalg.inv(M) @ x_novo).astype(int), "(volta à distribuição original)")

Soma de cada coluna de M: ['1', '1', '1', '1']
1^T · M == 1^T ? True
Todas as entradas de M >= 0 ? True
Total antes: 72  |  Total depois: 72
Nova alocação toda >= 0 e inteira ? True
  x=[ 4 38 32 21], soma=95 -> soma de Mx = 95.0000, min(Mx) = 5.750
  x=[21 42  4 34], soma=101 -> soma de Mx = 101.0000, min(Mx) = 7.517
  x=[10  4 26 48], soma=88 -> soma de Mx = 88.0000, min(Mx) = 3.800

det(M) = 0.8312  (invertível)
M^-1 · x_novo = [12 20 16 24] (volta à distribuição original)


# Resumo dos resultados

| Região | Atual | Necessário | Após redistribuição | Movimentação |
|---|---|---|---|---|
| Norte | 12 | 14 | 14 | recebe 2 do Oeste |
| Sul | 20 | 19 | 19 | cede 1 ao Leste |
| Leste | 16 | 18 | 18 | recebe 1 do Sul e 1 do Oeste |
| Oeste | 24 | 21 | 21 | cede 3 (2 Norte, 1 Leste) |
| **Total** | **72** | **72** | **72** | 4 ônibus movidos |

- **Etapa 1:** $\det(A) = 3 \ne 0$ ⇒ solução única $(12, 20, 16, 24)$, verificada em todos os registros.
- **Etapa 2:** a transformação $T(\mathbf{x}) = M\mathbf{x}$, com $M$ estocástica por colunas, leva a frota para $(14, 19, 18, 21)$, zerando os 160 passageiros sem vaga, conservando os 72 ônibus e sem gerar valores negativos.